# Bronze Ingestion - Microsoft Planetary Computer (multi-collection)

**Workload:** Geohazard demo - Bronze layer  
**Source:** [Microsoft Planetary Computer STAC API](https://planetarycomputer.microsoft.com/api/stac/v1)  
**Area of Interest (AOI):** Maple Ridge, British Columbia (south coast)

## Why this AOI
Maple Ridge is the temporary shared site for this run: every source notebook
anchors on the **same** lat/lon + radius so the satellite/elevation layers below
and the companion `bronze_bc_surficial_geology` notebook describe **the same
ground**, which is what a geohazard analysis needs. (Note: the BC surficial
geology dataset only covers the Interior Plateau, so it may return no features
at this coastal AOI - see that notebook for details.)

## What this notebook does
1. Defines a single AOI (lat/lon + radius -> bounding box).
2. Provides one reusable helper, `ingest_collection(...)`, that queries the STAC
   API and writes item **metadata** (not the rasters) into a bronze Delta table.
3. Ingests 7 complementary collections, each into its own bronze table.

> This is a **metadata bronze** pattern: we capture the STAC item records
> (ids, footprints, asset lists, properties). Downloading the underlying COGs is
> a later (silver/gold) concern and is intentionally out of scope here.

## Collections ingested
| STAC collection | Bronze table | Theme | Time filter |
| --- | --- | --- | --- |
| `io-lulc-9-class` | `bronze_io_lulc_9_class` | Land use / land cover | none (annual) |
| `esa-worldcover` | `bronze_esa_worldcover` | Land cover | none (epochs) |
| `cop-dem-glo-30` | `bronze_cop_dem_glo_30` | 30 m elevation (DEM) | none (static) |
| `sentinel-2-l2a` | `bronze_sentinel_2_l2a` | Optical imagery | 2024 summer, cloud < 30% |
| `sentinel-1-rtc` | `bronze_sentinel_1_rtc` | Radar (SAR) | 2024 summer |
| `alos-palsar-mosaic` | `bronze_alos_palsar_mosaic` | L-band radar mosaic | none (annual) |
| `hgb` | `bronze_hgb` | Above/below-ground biomass | none (static) |

## 1. Shared configuration and helper
Run this cell once. It defines the AOI, the bounding box, a stable bronze
schema, and the `ingest_collection()` function used by every collection cell
below.

In [ ]:
import math
import json
import requests
from datetime import datetime, timezone
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
)

# --- Area of Interest: Maple Ridge, BC -------------------------------------
LATITUDE = 49.2193
LONGITUDE = -122.5984
RADIUS_KM = 20
AOI_NAME = "Maple Ridge, BC"

STAC_SEARCH_URL = "https://planetarycomputer.microsoft.com/api/stac/v1/search"
SOURCE_API = "planetary_computer_stac"


def radius_to_bbox(lat, lon, radius_km):
    """Approximate a [min_lon, min_lat, max_lon, max_lat] box around a point."""
    lat_delta = radius_km / 111.32
    cos_lat = math.cos(math.radians(lat))
    lon_delta = 180.0 if abs(cos_lat) < 1e-8 else radius_km / (111.32 * cos_lat)
    return [lon - lon_delta, lat - lat_delta, lon + lon_delta, lat + lat_delta]


BBOX = radius_to_bbox(LATITUDE, LONGITUDE, RADIUS_KM)

# Explicit, stable schema so every collection lands in a consistently typed
# Delta table even when a particular property is missing for that collection.
BRONZE_SCHEMA = StructType([
    StructField("item_id", StringType(), True),
    StructField("collection", StringType(), True),
    StructField("datetime_utc", StringType(), True),
    StructField("bbox_minx", DoubleType(), True),
    StructField("bbox_miny", DoubleType(), True),
    StructField("bbox_maxx", DoubleType(), True),
    StructField("bbox_maxy", DoubleType(), True),
    StructField("asset_count", IntegerType(), True),
    StructField("asset_keys", StringType(), True),
    StructField("cloud_cover", DoubleType(), True),
    StructField("platform", StringType(), True),
    StructField("properties_json", StringType(), True),
    StructField("source_api", StringType(), True),
    StructField("query_lat", DoubleType(), True),
    StructField("query_lon", DoubleType(), True),
    StructField("query_radius_km", DoubleType(), True),
    StructField("query_datetime", StringType(), True),
    StructField("ingested_at_utc", TimestampType(), True),
])


def _num(v):
    """Best-effort float cast; returns None on failure."""
    try:
        return float(v)
    except (TypeError, ValueError):
        return None


def ingest_collection(collection, target_table, datetime_range=None,
                      max_items=12, query=None):
    """Query the Planetary Computer STAC API for one collection over the AOI
    bbox and write the item metadata to a bronze Delta table (overwrite).

    Parameters
    ----------
    collection : str       STAC collection id (e.g. 'sentinel-2-l2a').
    target_table : str     Destination bronze Delta table name.
    datetime_range : str   Optional STAC datetime filter 'start/end'.
    max_items : int        Maximum number of items to request.
    query : dict           Optional STAC `query` extension filter.
    """
    payload = {"collections": [collection], "bbox": BBOX, "limit": max_items}
    if datetime_range:
        payload["datetime"] = datetime_range
    if query:
        payload["query"] = query

    resp = requests.post(STAC_SEARCH_URL, json=payload, timeout=90)
    resp.raise_for_status()
    features = resp.json().get("features", [])

    # Static (mosaic / landcover) collections may ignore a datetime filter.
    # If a filtered search returns nothing, retry once with the bbox only.
    if not features and (datetime_range or query):
        retry = {"collections": [collection], "bbox": BBOX, "limit": max_items}
        resp = requests.post(STAC_SEARCH_URL, json=retry, timeout=90)
        resp.raise_for_status()
        features = resp.json().get("features", [])

    now = datetime.now(timezone.utc)
    rows = []
    for f in features:
        props = f.get("properties", {}) or {}
        b = f.get("bbox", [None, None, None, None]) or [None, None, None, None]
        assets = f.get("assets", {}) or {}
        rows.append((
            str(f.get("id")) if f.get("id") is not None else None,
            str(f.get("collection")) if f.get("collection") is not None else collection,
            str(props.get("datetime")) if props.get("datetime") is not None else None,
            _num(b[0]) if len(b) > 0 else None,
            _num(b[1]) if len(b) > 1 else None,
            _num(b[2]) if len(b) > 2 else None,
            _num(b[3]) if len(b) > 3 else None,
            int(len(assets)),
            ",".join(sorted(assets.keys())) if assets else None,
            _num(props.get("eo:cloud_cover")),
            str(props.get("platform")) if props.get("platform") is not None else None,
            json.dumps(props, default=str),
            SOURCE_API,
            float(LATITUDE),
            float(LONGITUDE),
            float(RADIUS_KM),
            datetime_range,
            now,
        ))

    df = spark.createDataFrame(rows, schema=BRONZE_SCHEMA)
    (df.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").saveAsTable(target_table))
    n = df.count()
    print(f"[{collection}] -> {target_table}: wrote {n} rows")
    return n


print("AOI:", AOI_NAME)
print("BBOX (min_lon, min_lat, max_lon, max_lat):", [round(c, 4) for c in BBOX])
print("Helpers ready: radius_to_bbox(), ingest_collection(), BRONZE_SCHEMA")

## 2. `io-lulc-9-class` - Esri 10 m Land Use / Land Cover
Annual 9-class land-cover product. Useful for masking water, built-up areas,
and vegetation when interpreting geohazard signals.

In [ ]:
ingest_collection("io-lulc-9-class", "bronze_io_lulc_9_class", max_items=12)

## 3. `esa-worldcover` - ESA WorldCover 10 m
Global land-cover epochs (2020, 2021). A second, independent land-cover
reference to cross-check `io-lulc-9-class`.

In [ ]:
ingest_collection("esa-worldcover", "bronze_esa_worldcover", max_items=12)

## 4. `cop-dem-glo-30` - Copernicus DEM 30 m
Global digital elevation model. Slope and elevation derived from this DEM are
core inputs for landslide / geohazard susceptibility.

In [ ]:
ingest_collection("cop-dem-glo-30", "bronze_cop_dem_glo_30", max_items=12)

## 5. `sentinel-2-l2a` - Sentinel-2 surface reflectance
10 m optical imagery. Filtered to the 2024 snow-free season and to scenes with
**less than 30% cloud cover** using the STAC `query` extension.

In [ ]:
ingest_collection(
    "sentinel-2-l2a",
    "bronze_sentinel_2_l2a",
    datetime_range="2024-06-01/2024-09-30",
    max_items=20,
    query={"eo:cloud_cover": {"lt": 30}},
)

## 6. `sentinel-1-rtc` - Sentinel-1 Radiometrically Terrain Corrected
C-band radar that sees through cloud and night. Filtered to the same 2024
summer window for temporal alignment with Sentinel-2.

In [ ]:
ingest_collection(
    "sentinel-1-rtc",
    "bronze_sentinel_1_rtc",
    datetime_range="2024-06-01/2024-09-30",
    max_items=20,
)

## 7. `alos-palsar-mosaic` - ALOS PALSAR annual mosaic
L-band radar penetrates vegetation better than C-band and is sensitive to
surface roughness and moisture - complementary to Sentinel-1.

In [ ]:
ingest_collection("alos-palsar-mosaic", "bronze_alos_palsar_mosaic", max_items=12)

## 8. `hgb` - Harmonized Global Biomass
Above- and below-ground biomass. A vegetation-load proxy that helps separate
vegetated slopes from bare/disturbed ground.

In [ ]:
ingest_collection("hgb", "bronze_hgb", max_items=12)

## 9. Verification
Confirm every bronze table exists and report its row count. A `MISSING` line
means that collection cell has not been run yet (or returned no items).

In [ ]:
tables = [
    "bronze_io_lulc_9_class",
    "bronze_esa_worldcover",
    "bronze_cop_dem_glo_30",
    "bronze_sentinel_2_l2a",
    "bronze_sentinel_1_rtc",
    "bronze_alos_palsar_mosaic",
    "bronze_hgb",
]
for t in tables:
    try:
        c = spark.table(t).count()
        print(f"{t:30s} {c:6d} rows")
    except Exception as e:
        print(f"{t:30s} MISSING ({e.__class__.__name__})")